# ForestSight AI -- Forest Detection from Satellite Imagery

**Binary Semantic Segmentation** using U-Net, Attention U-Net, and DeepLabV3+

| Item | Detail |
|------|--------|
| **Dataset** | [Forest Aerial Images for Segmentation](https://www.kaggle.com/datasets/quadeer15sh/augmented-forest-segmentation) |
| **Task** | Binary pixel-wise segmentation (forest vs non-forest) |
| **Models** | U-Net, Attention U-Net, DeepLabV3+ |
| **Framework** | PyTorch + Segmentation Models PyTorch |

## 1. Setup

In [ ]:
!pip install -q torch torchvision segmentation-models-pytorch albumentations \
    opencv-python-headless matplotlib seaborn numpy pandas scikit-learn \
    tqdm Pillow kagglehub grad-cam

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('.'))
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from tqdm.auto import tqdm
import kagglehub

from src.config import Config, seed_everything
from src.dataset import load_pairs, create_dataloaders, get_val_transforms
from src.models import MODEL_REGISTRY, build_model
from src.trainer import train_model, evaluate
from src.visualize import (
    denormalize, plot_training_curves, plot_comparison_bars,
    plot_predictions, plot_confusion_matrix
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

## 2. Configuration

In [ ]:
cfg = Config()
seed_everything(cfg.seed)
print(f'Device: {cfg.device} | AMP: {cfg.use_amp}')

## 3. Dataset Download

In [ ]:
dataset_path = Path(kagglehub.dataset_download('quadeer15sh/augmented-forest-segmentation'))
print(f'Dataset: {dataset_path}')

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(str(dataset_path), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'{"  " * (level+1)}{f}')
        if len(files) > 3:
            print(f'{"  " * (level+1)}... +{len(files)-3} more')

In [ ]:
pairs = load_pairs(dataset_path)
print(f'Total image-mask pairs: {len(pairs)}')

## 4. Exploratory Data Analysis

In [ ]:
import random
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Images, Masks, and Overlays', fontsize=16, fontweight='bold')

for col, idx in enumerate(random.sample(range(len(pairs)), 4)):
    img = cv2.imread(str(pairs[idx][0]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(pairs[idx][1]), cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[mask > 127] = (overlay[mask > 127] * 0.5 + np.array([0, 200, 0]) * 0.5).astype(np.uint8)

    axes[0, col].imshow(img); axes[0, col].set_title(f'Image {idx}'); axes[0, col].axis('off')
    axes[1, col].imshow(mask, cmap='Greens'); axes[1, col].set_title('Mask'); axes[1, col].axis('off')
    axes[2, col].imshow(overlay); axes[2, col].set_title('Overlay'); axes[2, col].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
heights, widths, forest_ratios = [], [], []
for img_p, mask_p in tqdm(pairs, desc='Analyzing'):
    img = cv2.imread(str(img_p))
    mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape[:2]
    heights.append(h); widths.append(w)
    forest_ratios.append((mask > 127).mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Distribution', fontsize=14, fontweight='bold')
axes[0].hist(heights, bins=20, color='steelblue', edgecolor='white'); axes[0].set_title('Height')
axes[1].hist(widths, bins=20, color='coral', edgecolor='white'); axes[1].set_title('Width')
axes[2].hist(forest_ratios, bins=30, color='forestgreen', edgecolor='white'); axes[2].set_title('Forest Ratio')
axes[2].axvline(np.mean(forest_ratios), color='red', ls='--', label=f'Mean: {np.mean(forest_ratios):.2f}')
axes[2].legend()
plt.tight_layout(); plt.show()

## 5. Data Pipeline

In [ ]:
train_loader, val_loader, test_loader, train_pairs, val_pairs, test_pairs = create_dataloaders(
    pairs, train_ratio=cfg.train_ratio, val_ratio=cfg.val_ratio,
    image_size=cfg.image_size, batch_size=cfg.batch_size,
    num_workers=cfg.num_workers, seed=cfg.seed,
)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

batch_img, batch_mask = next(iter(train_loader))
print(f'Batch shapes: img={batch_img.shape}, mask={batch_mask.shape}')

## 6. Model Overview

In [ ]:
for name, arch in MODEL_REGISTRY.items():
    m = build_model(arch)
    total = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'{name:20s} | {total:.1f}M params')
    del m

## 7. Train All Models

In [ ]:
results = {}
histories = {}

for model_name, architecture in MODEL_REGISTRY.items():
    print(f"\n{'='*60}\n  Training: {model_name}\n{'='*60}")
    model = build_model(architecture)
    model, history, best_iou = train_model(
        model, train_loader, val_loader,
        architecture=architecture, device=cfg.device, use_amp=cfg.use_amp,
        lr=cfg.lr, weight_decay=cfg.weight_decay, epochs=cfg.epochs,
        patience=cfg.early_stop_patience,
        dice_weight=cfg.dice_weight, bce_weight=cfg.bce_weight,
        checkpoint_dir=cfg.checkpoint_dir,
    )
    histories[model_name] = history
    results[model_name] = {'best_val_iou': best_iou}
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 8. Training Curves

In [ ]:
plot_training_curves(histories, save_path='training_curves.png')

## 9. Test Set Evaluation

In [ ]:
test_results = {}
for model_name, architecture in MODEL_REGISTRY.items():
    ckpt_path = cfg.checkpoint_dir / f'{architecture}_best.pth'
    model = build_model(architecture).to(cfg.device)
    ckpt = torch.load(ckpt_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_m = evaluate(model, test_loader, cfg.device, cfg.use_amp)
    test_results[model_name] = {
        'IoU': test_m['iou'], 'Dice': test_m['dice'],
        'Accuracy': test_m['accuracy'], 'Precision': test_m['precision'],
        'Recall': test_m['recall'],
    }
    del model

results_df = pd.DataFrame(test_results).T.round(4)
print(results_df.to_string())
best_model_name = results_df['IoU'].idxmax()
print(f'\nBest Model: {best_model_name} (IoU={results_df.loc[best_model_name, "IoU"]:.4f})')

In [ ]:
plot_comparison_bars(test_results, save_path='model_comparison.png')

## 10. Predictions Visualization

In [ ]:
best_arch = MODEL_REGISTRY[best_model_name]
best_model = build_model(best_arch).to(cfg.device)
ckpt = torch.load(cfg.checkpoint_dir / f'{best_arch}_best.pth', map_location=cfg.device, weights_only=True)
best_model.load_state_dict(ckpt['model_state_dict'])
best_model.eval()

plot_predictions(best_model, test_loader, cfg.device, save_path='predictions.png')

## 11. Confusion Matrix

In [ ]:
plot_confusion_matrix(best_model, test_loader, cfg.device, save_path='confusion_matrix.png')

## 12. Grad-CAM Visualization

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget

if 'deeplabv3' in best_arch:
    target_layers = [best_model.encoder.layer4[-1]]
else:
    target_layers = [list(best_model.encoder.children())[-1]]

sample_images, sample_masks = next(iter(test_loader))
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(f'Grad-CAM -- {best_model_name}', fontsize=16, fontweight='bold')

try:
    cam = GradCAM(model=best_model, target_layers=target_layers)
    for i in range(min(3, sample_images.shape[0])):
        img_t = sample_images[i:i+1].to(cfg.device)
        img_np = denormalize(sample_images[i]).permute(1, 2, 0).numpy()
        mask_np = sample_masks[i, 0].numpy()
        targets = [SemanticSegmentationTarget(0, torch.tensor(mask_np))]
        grayscale_cam = cam(input_tensor=img_t, targets=targets)
        cam_img = show_cam_on_image(img_np, grayscale_cam[0], use_rgb=True)
        with torch.no_grad():
            pred = (torch.sigmoid(best_model(img_t)).cpu()[0, 0] > 0.5).float().numpy()
        for ax in axes[i]: ax.axis('off')
        axes[i,0].imshow(img_np); axes[i,0].set_title('Input')
        axes[i,1].imshow(mask_np, cmap='Greens'); axes[i,1].set_title('Ground Truth')
        axes[i,2].imshow(cam_img); axes[i,2].set_title('Grad-CAM')
        axes[i,3].imshow(pred, cmap='Greens'); axes[i,3].set_title('Prediction')
except Exception as e:
    print(f'Grad-CAM failed: {e}')
    for i in range(min(3, sample_images.shape[0])):
        img_np = denormalize(sample_images[i]).permute(1, 2, 0).numpy()
        with torch.no_grad():
            pred = (torch.sigmoid(best_model(sample_images[i:i+1].to(cfg.device))).cpu()[0,0] > 0.5).float().numpy()
        for ax in axes[i]: ax.axis('off')
        axes[i,0].imshow(img_np); axes[i,0].set_title('Input')
        axes[i,1].imshow(sample_masks[i,0].numpy(), cmap='Greens'); axes[i,1].set_title('Ground Truth')
        axes[i,2].imshow(pred, cmap='Greens'); axes[i,2].set_title('Prediction')
plt.tight_layout(); plt.savefig('gradcam.png', dpi=150, bbox_inches='tight'); plt.show()

## 13. Single-Image Inference

In [ ]:
def predict(image_path, model, device, threshold=0.5):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original = image.copy()
    t = get_val_transforms()(image=image)
    tensor = t['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor)).cpu().squeeze().numpy()
    mask = (pred > threshold).astype(np.uint8)
    mask_r = cv2.resize(mask, (original.shape[1], original.shape[0]), interpolation=cv2.INTER_NEAREST)
    overlay = original.copy()
    green = np.zeros_like(overlay); green[:,:,1] = 255
    overlay[mask_r > 0] = (overlay[mask_r > 0] * 0.5 + green[mask_r > 0] * 0.5).astype(np.uint8)
    pct = mask_r.mean() * 100
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'Forest Coverage: {pct:.1f}%', fontsize=14, fontweight='bold')
    axes[0].imshow(original); axes[0].set_title('Input'); axes[0].axis('off')
    axes[1].imshow(mask_r, cmap='Greens'); axes[1].set_title('Mask'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.tight_layout(); plt.show()
    return pct

pct = predict(test_pairs[0][0], best_model, cfg.device)
print(f'Forest coverage: {pct:.1f}%')

## 14. Summary

Trained and compared 3 architectures for forest detection:
1. **U-Net** (EfficientNet-B4)
2. **Attention U-Net** (EfficientNet-B4 + scSE)
3. **DeepLabV3+** (ResNet-101 + ASPP)

All models use pretrained ImageNet encoders, Dice+BCE loss, AdamW, cosine annealing, and early stopping.

In [ ]:
print(results_df.to_string())
print(f'\nBest: {best_model_name} (IoU={results_df.loc[best_model_name, "IoU"]:.4f})')